# OJ-gen Unsloth 학습 -> 평가 -> 재학습 -> 패키징 순서

solved.ac 문제 난이도는 Bronze, Silver, Gold, Platinum, Diamond, Ruby 버킷과 tier value 1~30만 사용합니다. Master는 문제 난이도 버킷이 아니므로 생성 비율에서 제외합니다.

In [ ]:
%pip install -e .
%pip install -r requirements-train.txt

In [ ]:
MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
BASE_OUT = "outputs/qwen3_6_35b_a3b_v1"
MAX_SEQ_LENGTH = 8192
TARGET_TOTAL = 50000
MIN_LOCAL_SCORE = 0.90
MIN_LLM_SCORE = 0.85
MIN_CONFIDENCE = 0.75

## 1. SFT 데이터 생성 및 형식 점검

In [ ]:
!python -m ojgen.build_sft_from_csv \
  --csv data/raw/boj_problems_structured.csv \
  --out data/sft/full.jsonl \
  --no-require-samples \
  --language-filter ko \
  --inspect

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="data/sft/full.jsonl", split="train")
print(ds)
print(type(ds[0]["messages"]), ds[0]["messages"][0])

## 2. 1차 Unsloth 학습

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m ojgen.train_sft \
  --data data/sft/full.jsonl \
  --out {BASE_OUT} \
  --model {MODEL_ID} \
  --max-seq-length {MAX_SEQ_LENGTH} \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum 8 \
  --lora-r 32 \
  --lora-alpha 64 \
  --target-modules all \
  --no-packing

## 3. solved.ac 난이도별 동일 비율 생성 프롬프트 작성

In [ ]:
!python generate_prompts.py \
  --out data/prompts_50k.jsonl \
  --total {TARGET_TOTAL} \
  --seed 3407

## 4. 후보 문제 생성

In [ ]:
!python -m ojgen.generate \
  --adapter {BASE_OUT} \
  --base-model {MODEL_ID} \
  --prompts data/prompts_50k.jsonl \
  --out outputs/generated_v1.jsonl \
  --max-seq-length {MAX_SEQ_LENGTH} \
  --max-new-tokens 2048 \
  --temperature 0.8 \
  --top-p 0.95 \
  --num-return-sequences 1

## 5. 로컬 검증 + LLM 자체 평가

In [ ]:
!python -m ojgen.self_eval \
  --generated outputs/generated_v1.jsonl \
  --out outputs/evaluated_v1.jsonl \
  --judge-model {MODEL_ID} \
  --tensor-parallel-size 1 \
  --max-model-len {MAX_SEQ_LENGTH} \
  --temperature 0.0

## 6. 기준 미달이면 평가 통과 샘플로 재학습 데이터 증류

In [ ]:
!python -m ojgen.distill_evaluated_to_sft \
  --evaluated outputs/evaluated_v1.jsonl \
  --out data/sft/generated_accepted_v1.jsonl \
  --min-local-score {MIN_LOCAL_SCORE} \
  --min-llm-score {MIN_LLM_SCORE} \
  --min-confidence {MIN_CONFIDENCE} \
  --require-llm-eval

In [ ]:
from pathlib import Path
sources = [Path("data/sft/full.jsonl"), Path("data/sft/generated_accepted_v1.jsonl")]
out = Path("data/sft/full_plus_generated_v2.jsonl")
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as target:
    for source in sources:
        if source.exists():
            target.write(source.read_text(encoding="utf-8"))
print(out)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m ojgen.train_sft \
  --data data/sft/full_plus_generated_v2.jsonl \
  --out outputs/qwen3_6_35b_a3b_v2 \
  --model {MODEL_ID} \
  --max-seq-length {MAX_SEQ_LENGTH} \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum 8 \
  --lora-r 32 \
  --lora-alpha 64 \
  --target-modules all \
  --no-packing

## 7. 기준을 통과한 문제만 난이도별 동일 개수로 Markdown/zip 생성

In [ ]:
!python -m ojgen.package_md \
  --evaluated outputs/evaluated_v1.jsonl \
  --out-dir final_dataset \
  --zip oj_problems_balanced.zip \
  --min-local-score {MIN_LOCAL_SCORE} \
  --min-llm-score {MIN_LLM_SCORE} \
  --min-confidence {MIN_CONFIDENCE} \
  --require-llm-eval